# Google Earth Engine → 100-site feature matrix (Colab)

**No raster downloads.** Earth Engine computes site summaries on its servers; only the small result table is transferred to the notebook and downloaded as CSV.

**Input:** `sites.csv` with `site_id,latitude_deg,longitude_deg` (also accepts `lat/lon`, `lat/lng`, or `latitude/longitude`). **Output:** `gee_features.csv`.

**Scientific definitions:** `vegetation_cover_pct` is an NDVI-based *proxy*, `surface_thermal_inertia` is actually **day–night land-surface temperature difference (°C)**, and `surface_ice_fraction_pct` is a **MODIS NDSI snow-cover indicator**, *not measured glacier/ice coverage*. Spatial buffers of 100 m cannot increase the native resolution of coarse datasets.


In [ ]:
%pip -q install --upgrade earthengine-api pandas numpy

## 1. Authenticate and configure

Enter your **Google Cloud project ID** (registered for Earth Engine). Run this cell and follow the Google sign-in prompt.

In [ ]:
import ee
from google.colab import auth

PROJECT_ID = "YOUR_GCP_PROJECT_ID"  # <-- replace with your own
START_DATE = "2024-01-01"
END_DATE = "2025-01-01"  # exclusive; full year 2024
BUFFER_M = 100
OUTPUT_CSV = "gee_features.csv"

if PROJECT_ID == "YOUR_GCP_PROJECT_ID":
    raise ValueError("Please replace YOUR_GCP_PROJECT_ID with your Google Cloud Project ID.")

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized.")

## 2. Upload site CSV from your computer

Click **Choose Files**. This version reads the file you upload to the Colab session, instead of assuming it exists in a local directory.

In [ ]:
from google.colab import files
import io
import pandas as pd
import numpy as np

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one CSV file containing site coordinates.")
filename, contents = next(iter(uploaded.items()))
if not filename.lower().endswith('.csv'):
    raise ValueError("The uploaded file must be a .csv file.")

sites = pd.read_csv(io.BytesIO(contents), dtype={"site_id": "string"})
sites.columns = sites.columns.str.strip()
aliases = {
    "lat": "latitude_deg", "latitude": "latitude_deg",
    "lon": "longitude_deg", "lng": "longitude_deg", "longitude": "longitude_deg"
}
sites = sites.rename(columns={k:v for k,v in aliases.items() if k in sites.columns and v not in sites.columns})
required = ["site_id", "latitude_deg", "longitude_deg"]
missing = set(required) - set(sites.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}; expected {required}")
sites = sites[required].copy()
sites["site_id"] = sites["site_id"].astype("string").str.strip()
for c in ["latitude_deg", "longitude_deg"]:
    sites[c] = pd.to_numeric(sites[c], errors="raise")
if sites[required].isna().any().any() or sites["site_id"].eq("").any():
    raise ValueError("Empty site_id or missing coordinates detected.")
if sites.site_id.duplicated().any():
    raise ValueError("site_id values must be unique.")
if not sites.latitude_deg.between(-90, 90).all() or not sites.longitude_deg.between(-180, 180).all():
    raise ValueError("Invalid latitude or longitude range.")
print(f"Loaded {len(sites)} sites (target: 100).")
display(sites.head())

## 3. Build server-side data products

The products are MOD13Q1 (250 m), MOD11A2 (1 km), ERA5-Land MONTHLY_AGGR (~11 km), and MOD10A1 (500 m). Annual averages use valid observed scenes; missing snow/cloud/water retrievals remain masked rather than assumed zero.


In [ ]:
def site_collection(data):
    return ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Point([float(r.longitude_deg), float(r.latitude_deg)]).buffer(BUFFER_M),
            {"site_id": str(r.site_id)}
        ) for r in data.itertuples(index=False)
    ])

site_fc = site_collection(sites)

def collection(dataset):
    return ee.ImageCollection(dataset).filterDate(START_DATE, END_DATE)

# MOD13Q1 NDVI, scaled by 0.0001. QA flags 0 and 1 accepted.
def valid_ndvi(img):
    good = img.select('DetailedQA').bitwiseAnd(3).lte(1)
    return img.select('NDVI').multiply(0.0001).updateMask(good).rename('ndvi')

ndvi_mean = collection('MODIS/061/MOD13Q1').map(valid_ndvi).mean()
# Empirical NDVI-based vegetation fraction proxy; NOT ground-truth percent cover.
vegetation = (ndvi_mean.subtract(0.2).divide(0.6).clamp(0, 1)
              .pow(2).multiply(100).rename('vegetation_cover_pct'))

# Mean of paired (day-night) 8-day LST differences; *not physical thermal inertia*.
def paired_lst_delta(img):
    good_day = img.select('QC_Day').bitwiseAnd(3).lte(1)
    good_night = img.select('QC_Night').bitwiseAnd(3).lte(1)
    delta = img.select('LST_Day_1km').subtract(img.select('LST_Night_1km')).multiply(0.02)
    return delta.updateMask(good_day.And(good_night)).rename('surface_thermal_inertia')

thermal = collection('MODIS/061/MOD11A2').map(paired_lst_delta).mean()

# ERA5-Land: volumetric soil water layer 1, 0–7 cm, m³/m³.
soil = (collection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
        .select('volumetric_soil_water_layer_1').mean()
        .clamp(0,1).rename('soil_moisture_fraction'))

# MODIS NDSI snow cover (0–100); masks non-valid retrievals.
def valid_snow(img):
    band = img.select('NDSI_Snow_Cover')
    return band.updateMask(band.gte(0).And(band.lte(100))).rename('surface_ice_fraction_pct')

snow = collection('MODIS/061/MOD10A1').map(valid_snow).mean()

jobs = [
    ('vegetation_cover_pct', vegetation, 250),
    ('surface_thermal_inertia', thermal, 1000),
    ('soil_moisture_fraction', soil, 11132),
    ('surface_ice_fraction_pct', snow, 500),
]
print('Server-side products configured.')

## 4. Aggregate buffered sites on GEE and download **only numerical summaries**

`reduceRegions` executes on the EE server. Each query transfers only `site_id` and an aggregated mean, not raster pixels. Failed queries are retried with exponential backoff.

In [ ]:
import time

FEATURES = [j[0] for j in jobs]

def ee_values(image, feature_name, scale, n_attempts=4):
    reduced = image.reduceRegions(
        collection=site_fc,
        reducer=ee.Reducer.mean(),
        scale=scale,
        tileScale=4,
    )
    tiny = reduced.map(lambda f: ee.Feature(None, {
        'site_id': f.get('site_id'),
        feature_name: f.get('mean'),
    }))
    for attempt in range(n_attempts):
        try:
            data = tiny.getInfo()['features']
            break
        except Exception as exc:
            if attempt == n_attempts-1:
                raise RuntimeError(f'GEE extraction failed for {feature_name}') from exc
            delay = 2**attempt * 2
            print(f'  Retry {attempt+1}/{n_attempts-1} after {delay}s: {exc}')
            time.sleep(delay)
    out = pd.DataFrame([f['properties'] for f in data])
    if len(out) != len(sites) or out.site_id.duplicated().any():
        raise RuntimeError(f'Unexpected site count / duplicates for {feature_name}')
    return out[['site_id', feature_name]]

result = sites.copy()
for feature_name, image, scale in jobs:
    print(f'Extracting {feature_name} ...')
    table = ee_values(image, feature_name, scale)
    result = result.merge(table, on='site_id', how='left', validate='one_to_one')
    print(f'  done; missing: {result[feature_name].isna().sum()} / {len(result)}')

display(result.head())

## 5. Clean and export CSV

Raw missingness is recorded in `*_was_missing`. Median imputation is used only for features with at least one valid observation. For scientifically rigorous ML, consider fitting imputation and normalization on **training data only**; this notebook provides a convenient single-file baseline.

In [ ]:
for col in FEATURES:
    result[col] = pd.to_numeric(result[col], errors='coerce').replace([np.inf,-np.inf], np.nan)

for col in ['vegetation_cover_pct', 'surface_ice_fraction_pct']:
    result[col] = result[col].clip(0, 100)
result['soil_moisture_fraction'] = result['soil_moisture_fraction'].clip(0, 1)

print('Missing values BEFORE imputation:')
print(result[FEATURES].isna().sum().to_string())

for col in FEATURES:
    result[f'{col}_was_missing'] = result[col].isna().astype('int8')
    if result[col].isna().any():
        med = result[col].median()
        if pd.isna(med):
            raise ValueError(f'All sites lack valid values for {col}; change dates or data source.')
        result[col] = result[col].fillna(med)
        print(f'Imputed {col} with median {med:.4f}')

result['vegetation_cover_fraction'] = result['vegetation_cover_pct'] / 100.0
result['surface_ice_fraction'] = result['surface_ice_fraction_pct'] / 100.0
x = result['surface_thermal_inertia']
result['thermal_difference_scaled_01'] = ((x-x.min())/(x.max()-x.min())
                                          if x.max() > x.min() else 0.5)

assert not result[FEATURES].isna().any().any()
assert len(result) == len(sites)
result.to_csv(OUTPUT_CSV, index=False, float_format='%.6f')
print(f'Saved {OUTPUT_CSV} with shape {result.shape}')
display(result.head())

# Automatically download the small feature table to your computer.
files.download(OUTPUT_CSV)

## Important notes

- 100 m refers to **geometry of aggregation**, not output spatial resolution. Coarser products represent their surrounding pixel / grid cell.
- `surface_ice_fraction_pct` is a legacy feature name for compatibility: scientifically it is mean **NDSI snow-cover indicator**; it is not glacier fractional area.
- `surface_thermal_inertia` is a legacy name for **LST day–night difference** in °C; it can be negative and does not intrinsically lie in [0, 1].
- If a feature is missing for all sites, the notebook stops rather than silently creating misleading zeros.
- [MOD13Q1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13Q1), [MOD11A2](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD11A2), [ERA5-Land](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR), [MOD10A1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD10A1).
